# Estrazione dei record di tracciabilità (regex v2)

Estrae dal dataset `technology_mapping` (aiuti di stato 2014–2025, ~24M record) i progetti che
parlano di **tracciabilità**, applicando la regex a **tre campi**: `TITOLO_MISURA`,
`TITOLO_PROGETTO`, `DESCRIZIONE_PROGETTO`.

## Perché una v2: due difetti indipendenti

### 1. Difetto di *design* — co-occorrenza senza prossimità
Le regole composte della v1 verificavano che due termini comparissero **entrambi nel testo, a
qualsiasi distanza**. Misurato sui match reali: i due termini distavano **mediana 148 caratteri
(42% oltre 200)** su descrizioni lunghe ~1.000 caratteri. Non una relazione semantica: una
collisione lessicale. Con verbi generici (`monitoraggio` 1.831 hit, `identificazione` 595)
accoppiati a un qualsiasi `prodotti`, venivano etichettate come "tracciabilità" consulenze di
cybersecurity, librerie online e gestionali di magazzino di ristoranti.
**Solo il 18,2% dei positivi v1 conteneva un vero termine di tracciabilità.**

### 2. Difetto di *ambiente* — `\b` ASCII-only in pandas 3.0
pandas ≥ 3.0 usa stringhe **Arrow-backed**, che per le regex passano da **RE2**, dove `\b` e `\w`
sono **ASCII-only**. Dopo una lettera accentata il word-boundary non esiste:

```python
pd.Series(["tracciabilità:"]).str.contains(r'\btracciabilit[aà]\b')  # False  (Arrow/RE2)
re.search(r'\btracciabilit[aà]\b', "tracciabilità:")                 # True   (re)
```

La regex v1 **era corretta quando fu scritta** (pandas ≤ 2.x, motore `re`): si è rotta in silenzio
con l'upgrade di pandas, perdendo **~2.000 record che contengono esplicitamente la parola
"tracciabilità"**. `traceability_patterns._normalize` forza `object` dtype → motore `re`,
con semantica Unicode piena.

> Questi due difetti spiegano anche il fallimento del BERT: era addestrato su queste label
> (`prepare_traceability_data.py` importa `get_mask`), quindi distillava un maestro rumoroso, e la
> sua validazione contro lo stesso oracolo era **circolare**.

## Cosa cambia nella v2 (`traceability_patterns.py`)

1. **Prossimità**: `near(A, B, window)` richiede i due termini entro ~una proposizione.
2. **Niente verbi generici**: `monitoraggio`/`identificazione`/`autenticazione` sono fuori.
3. **Falsi amici esclusi**: `marcatura CE` (conformità di prodotto, non tracciabilità).
4. **Etichettatura/marcatura** valgono solo in contesto di filiera (finestra stretta).
5. **Engine deterministico**: `object` dtype → `re` Unicode-aware.
6. Ogni positivo espone `MATCH_SOURCE` (quale campo) e `MATCH_RULE` (quale regola) → **spiegabile**.

> `TITOLO_MISURA` è il nome dello *schema di aiuto* ("Fondo di garanzia per le PMI"), non del
> progetto: un match lì è evidenza di *programma*, più debole. Una singola misura può coprire >100k
> record, perciò `MATCH_SOURCE` permette di isolarne (ed escluderne) il contributo.

## 1. Test battery

Prima di rilanciare la pipeline, la regex deve superare i casi noti. I falsi positivi qui sotto
**non sono inventati**: sono record reali che la v1 classificava come tracciabilità.

In [ ]:
import sys

import pandas as pd

sys.path.insert(0, '.')
from traceability_patterns import get_mask, rule_labels, rule_masks

TRUE_POS = [
    "Tracciabilita - Sicurezza",
    "Sistemi di gestione qualità e tracciabilità: garanzia nel settore alimentare",  # accento + ':'
    "Sviluppo di una piattaforma blockchain per la provenienza del prodotto",
    "Sistema infotecnologico basato su blockchain per la filiera dei prodotti alimentari (BLOCKFIL)",
    "MyLime: piattaforma per i dati della filiera automotive, certificati con Blockchain",
    "Sistema di etichettatura dei lotti di produzione con QR code per la provenienza",
    "Monitoraggio della filiera agroalimentare tramite sensori IoT",
    "Progetto di formazione sulla rintracciabilita' alimentare",
    "Tracciamento dei lotti di materie prime lungo la filiera produttiva",
]

# Falsi positivi REALI della v1 (estratti dai suoi match sul dataset)
FALSE_POS = [
    "CONSULENZA IN CYBERSECURITY, con identificazione dei prodotti critici e analisi dei rischi",
    "Libreria online: e-commerce che diffonde libri e prodotti, con monitoraggio delle vendite",
    "Informatizzazione del magazzino del ristorante, con monitoraggio delle giacenze e dei prodotti",
    "Misurazione e monitoraggio dei livelli di campo elettromagnetico e di prodotti software",
    "Wearable Technologies - Internet of Things",
    "raccontare la filiera corta ad Arezzo",
    "Produzione integrata di pellet e biochar italiani di qualita' in filiera corta",
    "Realizzare le attivita' necessarie per la marcatura CE del prodotto UMR-BT",
    "Investimento per il controllo di gestione delle commesse e il monitoraggio della produzione",
    "Standard SA8000 di responsabilita' sociale, con controllo dei fornitori e delle materie prime",
]

s = pd.Series(TRUE_POS + FALSE_POS)
m = get_mask(s)
lab = rule_labels(rule_masks(s), index=s.index)

ok = True
print("=== VERI POSITIVI (attesi: MATCH) ===")
for i, txt in enumerate(TRUE_POS):
    good = bool(m[i]); ok &= good
    print(f"  {'PASS' if good else 'FAIL'} [{lab[i] or '-':<22}] {txt[:58]}")

print("\n=== FALSI POSITIVI della v1 (attesi: NO MATCH) ===")
off = len(TRUE_POS)
for i, txt in enumerate(FALSE_POS):
    good = not bool(m[off + i]); ok &= good
    print(f"  {'PASS' if good else 'FAIL'} [{lab[off + i] or '-':<22}] {txt[:58]}")

assert ok, "La test battery non passa: non lanciare la pipeline."
print("\n*** TEST BATTERY: TUTTI PASS ***")

## 2. Estrazione (multiprocessing, 10 processi)

La logica pesante sta in `traceability_worker.py` (modulo esterno): evita l'errore di pickling di
Jupyter e permette il `Pool`.

Due dettagli non ovvi:
- **`N_PROC = 10`** — 12 core fisici meno 2 lasciati liberi.
- **contesto `fork`** — Python 3.14 usa `forkserver` di default, che re-importa `__main__` e quindi
  *non funziona* dentro un notebook (`RuntimeError: An attempt has been made to start a new
  process before the current process has finished its bootstrapping phase`).

In [ ]:
import glob
import multiprocessing as mp
import os
import time

from traceability_worker import process_file

INPUT_DIR = '../../data/technology_mapping'
OUTPUT_DIR = '../../data/traceability'
N_PROC = 10  # 12 core fisici - 2 liberi

os.makedirs(OUTPUT_DIR, exist_ok=True)
files = sorted(glob.glob(os.path.join(INPUT_DIR, 'reclassified_multiclass_aiuti_*.csv')))
print(f"File da elaborare: {len(files)} | Pool: {N_PROC} processi")

t0 = time.time()
ctx = mp.get_context('fork')  # Python 3.14: il default forkserver non gira da notebook
with ctx.Pool(processes=N_PROC) as pool:
    results = pool.map(process_file, [(f, OUTPUT_DIR) for f in files])

print("\n--- Riepilogo ---")
totale = 0
for filename, matches in sorted(results):
    print(f"{filename}: {matches} record estratti.")
    totale += matches
print(f"\nTOTALE: {totale} record  (in {time.time() - t0:.0f}s)")

## 3. Audit: da dove viene ogni etichetta

`MATCH_SOURCE` = quale campo ha fatto match · `MATCH_RULE` = quale regola è scattata.

Due domande a cui questa cella risponde:
- *quanti positivi dipendono solo dal nome della misura?* (evidenza di programma, non di progetto)
- *quanti hanno un termine forte di tracciabilità e quanti solo una regola composta?*
  Nella v1 l'81,8% viveva di sole regole composte — era il sintomo del problema.

In [ ]:
df = pd.concat(
    [pd.read_csv(f, low_memory=False)
     for f in sorted(glob.glob(os.path.join(OUTPUT_DIR, 'traceability_aiuti_*.csv')))],
    ignore_index=True,
)
print(f"Record totali: {len(df):,} | descrizioni uniche: {df['DESCRIZIONE_PROGETTO'].nunique():,}\n")

print("=== MATCH_SOURCE (quale campo) ===")
print(df['MATCH_SOURCE'].value_counts().to_string())

print("\n=== MATCH_RULE (quale regola) ===")
print(df['MATCH_RULE'].value_counts().head(10).to_string())

strong = df['MATCH_RULE'].str.contains('strong')
print(f"\nCon termine FORTE di tracciabilità : {strong.sum():>6,} ({strong.mean():.1%})   [v1: 18,2%]")
print(f"Solo regole composte               : {(~strong).sum():>6,} ({(~strong).mean():.1%})   [v1: 81,8%]")

misura = df['MATCH_SOURCE'].str.contains('titolo_misura')
solo_misura = df['MATCH_SOURCE'] == 'titolo_misura'
print(f"\nCoinvolgono TITOLO_MISURA          : {misura.sum():>6,} ({misura.mean():.2%})")
print(f"Positivi SOLO per TITOLO_MISURA    : {solo_misura.sum():>6,} ({solo_misura.mean():.2%})")
print("  -> se fosse alta, i positivi sarebbero dominati dal nome dello schema di aiuto")
print("     invece che dal contenuto del progetto: qui si potrebbe decidere di escluderli.")

## 4. Delta v1 → v2 (stesso corpus, stessa passata, stesso engine)

I CSV v1 su disco erano stati generati tempo fa con corpus/regex non più allineati: confrontarli con
l'output v2 sarebbe apples-to-oranges. `compare_regex_versions.py` ricalcola **entrambe** le
maschere sugli stessi record, nella stessa passata, **con lo stesso motore regex** — così il delta
misura la differenza di *design*, non il bug di engine di pandas 3.0.

In [ ]:
!../../.venv/bin/python compare_regex_versions.py

## 5. Prossimo passo: validazione NON circolare

I numeri qui sopra dicono *quanto* la v2 cambia, non *se ha ragione*. Per saperlo serve
annotazione umana — è esattamente il passo che mancava e che faceva sembrare buono il BERT.

```bash
python build_regex_gold_set.py     # -> data/traceability/validation/regex_gold_set.csv
# compilare a mano la colonna GOLD_LABEL: 'tracciabilita' | 'altro'
python score_gold_set.py ../../data/traceability/validation/regex_gold_set.csv
```

Il gold set è stratificato in tre strati, ciascuno con una domanda precisa:

| Strato | Domanda |
|---|---|
| **A** positivi v2 | **Precision**: quanti positivi v2 sono veri? |
| **B** positivi v1 scartati da v2 | Gli scartati erano davvero falsi positivi? |
| **C** near-miss non catturati | **Recall**: quanti veri positivi la v2 si perde? |

---
## Legacy — dataset di training BERT (percorso PARCHEGGIATO)

Le celle originali costruivano il dataset per il modello BERT. Sono state **rimosse dal flusso
attivo**: generavano le label a partire dalla regex stessa, cioè l'oracolo circolare che rendeva la
validazione priva di valore (`F1 = 0.928` misurato contro il proprio maestro).

Il codice resta in git (`git log -- traceability_extraction.ipynb`). Se un giorno si torna a un
modello, va addestrato su **annotazioni umane** (il gold set), non sull'output della regex —
altrimenti il modello non può, per costruzione, superare il suo maestro.